## Configuration

In [1]:
from graph_reader import GraphReader
from pipeline import Pipeline
from compiler import LdioCompiler, RdfcCompiler
from pyshacl import validate
from rdflib import Graph
import pandas as pd

input_folder = "C:\\Users\\ThomasCarsten\\OneDrive\\Projects\\Dishacled\\pipeline generator prototype\\data\\"
output_folder = "C:\\Users\\ThomasCarsten\\OneDrive\\Projects\\Dishacled\\pipeline generator prototype\\out\\"

## Load a pipeline

In [11]:
# Initializing the graph reader and reading in the graph
graph_reader = GraphReader()
graph_reader.read_graph(input_folder, ["component catalogue.ttl", "pipeline.ttl"])

# Initializing a new Pipeline Data class and loading the data of the LDIO pipeline
ldio_pipeline = Pipeline()
ldio_pipeline.load_pipeline(":LdioExamplePipeline", graph_reader)

# Doing the same with the rdfc_pipeline
rdfc_pipeline = Pipeline()
rdfc_pipeline.load_pipeline(":RdfcExamplePipeline", graph_reader)

## validate a pipeline

In [12]:
# Validating the pipeline
results = validate(
      data_graph=ldio_pipeline.graph, # Shapes are contained in the data graph
      inference='none',
      abort_on_first=False,
      allow_infos=False,
      allow_warnings=True,
      meta_shacl=False,
      advanced=True,
      js=False,
      debug=False)

conforms, results_graph, results_text = results
print(results_text)

Validation Report
Conforms: True



## Compile a LDIO pipeline

#### LDIO

In [13]:
# Compiling an LDIO config file based on the information contained in the pipeline
ldio_compiler = LdioCompiler()

# You have to compile and save the config file first before the dockerfile if you use a non-default name for the config file. 
# This is because the dockerfile refers to the config file and hence has to look up its filename first.
ldio_compiler.compile_config(ldio_pipeline)
ldio_compiler.save_config(output_folder + "ldio_example_pipeline.yml")
ldio_compiler.compile_dockerfile(ldio_pipeline)
ldio_compiler.save_dockerfile(output_folder + "docker-compose.yml")


## Compile a RDF Connect pipeline

In [14]:
rdfc_compiler = RdfcCompiler()
print(rdfc_compiler.compile_config(rdfc_pipeline))

@base <http://myproject.local/> .
@prefix : <http://example.org/example/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfc: <https://w3id.org/rdf-connect#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

:RdfcConsoleOutStep a rdfc:LogProcessorJs ;
    owl:imports <node_modules/@rdfc/log-processor-ts/processor.ttl> ;
    rdfc:label "test" ;
    rdfc:level "debug" ;
    rdfc:reader :channel_1 .

:RdfcExamplePipeline a rdfc:Pipeline ;
    rdfc:consistsOf :env .

:RdfcLdesClientStep a rdfc:LdesClient ;
    owl:imports <node_modules/ldes-client/processor.ttl> ;
    rdfc:follow true ;
    rdfc:output :channel_1 ;
    rdfc:url <https://ca-westtoerwin-nginx-prod.livelyisland-1fa58ea1.westeurope.azurecontainerapps.io/touristattractions/latestView> .

:env rdfc:instantiates rdfc:NodeRunner ;
    rdfc:processor rdfc:LdesClient,
        rdfc:LogProcessorJs .

rdfc:NodeRunner owl:imports <node_modules/@rdfc/js-runner/index.ttl> .

:channel_1 a rdfc:Reader,
        rdfc:Writer .
